In [825]:
import numpy as np
import csv
import pandas as pd
from itertools import islice
import json
import missingno as msno
import matplotlib.pyplot as plt

In [826]:
books = pd.read_csv('../data/final_book_dataset.csv', delimiter='\t')

/var/folders/bt/8s6v7ngs6m93f2x58bpj1zl00000gn/T/ipykernel_91165/2359748259.py:1: DtypeWarning: Columns (0: isbn) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv('../data/final_book_dataset.csv', delimiter='\t')


In [827]:
books.head()

,Unnamed: 0,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,tags,hugo,locus
0,7649,9372.0,The Long Loud Silence,Wilson Tucker,1954,1954-00-00,Dell,1914.0,"Deer Creek, Illinois, USA",0899683754,Publisher's description: Tomorrow's war -- the...,"['Anatomy of Wonder 1 Core Collection', 'biolo...",False,False
1,82380,2215181.0,Mrs. Candy Strikes It Rich,Robert Tallant,1954,1954-00-00,Doubleday,1909.0,"New Orleans, Louisiana, USA",NaN,NaN,NaN,False,False
2,53874,1392436.0,Return to the Lost Planet,Angus MacVicar,1954,1954-00-00,Burke,1908.0,"Duror, Argyll, Scotland, UK",NaN,NaN,NaN,False,False
3,41661,1112206.0,Rainbow on the Road,Esther Forbes,1954,1954-00-00,Houghton Mifflin,1891.0,"Westborough, Massachusetts, USA",NaN,NaN,NaN,False,False
4,1728,1908.0,The Forgotten Planet,Murray Leinster,1954,1954-00-00,Ace Books,1896.0,"Norfolk, Virginia, USA",0881846163,<b>From the first page of the Ace Double:</b> ...,"['insects', 'Librivox', 'Project Gutenberg', '...",False,False


In [828]:
books['author'].value_counts()

author
R. L. Stine                     329
Michael Anderle                 281
Eve Langlais                    202
Odette C. Bell                  202
Michael Anderle, Martha Carr    197
                               ... 
West Ambrose                      1
Madeline Bell                     1
Leslie Adame                      1
Maddie Martinez                   1
Samantha Browning Shea            1
Name: count, Length: 40019, dtype: int64

In [829]:
books[books.duplicated(['title', 'author'], keep=False)].sort_values(by='title')

,Unnamed: 0,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,tags,hugo,locus


In [830]:
books_eda = books[books['release_year'] <= 1980]

In [144]:
finalists = books_eda[books['hugo'] | books['locus']]
nonfinalists = books_eda[~(books['hugo'] | books['locus'])]

/var/folders/bt/8s6v7ngs6m93f2x58bpj1zl00000gn/T/ipykernel_91165/2803869070.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  finalists = books_eda[books['hugo'] | books['locus']]
/var/folders/bt/8s6v7ngs6m93f2x58bpj1zl00000gn/T/ipykernel_91165/2803869070.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  nonfinalists = books_eda[~(books['hugo'] | books['locus'])]


In [145]:
test_tags = books_eda['tags'][0]

### Publisher Info

In [146]:
finalists['first_publisher'].value_counts().sort_index()

first_publisher
Ace Books               27
Ace Books / SFBC         1
Ace Fantasy Books        1
Arkham House             1
Arrow Books              4
                        ..
Warner Books             1
Warner Books / SFBC      1
Weybright and Talley     1
Windrush                 1
ibooks                   1
Name: count, Length: 102, dtype: int64

Publisher will be a useful feature, but currently many publishers have slightly different names!

In [147]:
def clean_publishers(publisher):
    # "/" separates an imprint/collection from its parent company, keep only imprint/collection:
    pub = str(publisher).split("/")[0].strip()

    # turn to lowercase
    pub = pub.lower()

    common_publishers = ['tor', 'ace', 'del rey', 'doubleday', 'bloomsbury', 'orbit', 'gollancz', 'daw', 'harpercollins', 'macmillan', 'simon & schuster']
    for cp in common_publishers:
        if cp in pub:
            # print(cp, pub)
            pub = cp

    # remove bracketed items like (us) (uk)
    # pub = pub.split("(")[0].strip()

    # # remove some filler words that cause differences between same publishers
    # removals = [" science fiction", ' fantasy', ' uk', ' us', '(uk)', '(us)','.com', ' press']
    # for remove in removals:
    #     pub = pub.replace(remove, '')
    return pub.replace(' ', '')

In [152]:
books_eda['first_publisher'] = books_eda['first_publisher'].apply(clean_publishers)

In [148]:
finalists['first_publisher'] = finalists['first_publisher'].apply(clean_publishers)
nonfinalists['first_publisher'] = nonfinalists['first_publisher'].apply(clean_publishers)

# finalists2.value_counts().sort_index()

In [149]:
finalists['first_publisher'].value_counts(normalize=True)#.reset_index(name='counts').set_index('first_publisher').plot.pie(y='counts', legend=False)

first_publisher
ace                0.095238
delrey             0.066667
daw                0.057143
ballantinebooks    0.053968
doubleday          0.044444
                     ...   
unwin              0.003175
summitbooks        0.003175
gauntletpress      0.003175
arkhamhouse        0.003175
kinnell            0.003175
Name: proportion, Length: 87, dtype: float64

In [157]:
books[~books['tags'].isna()]

,Unnamed: 0,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,tags,hugo,locus
0,7649,9372.0,The Long Loud Silence,Wilson Tucker,1954,1954-00-00,Dell,1914.0,"Deer Creek, Illinois, USA",0899683754,Publisher's description: Tomorrow's war -- the...,"['Anatomy of Wonder 1 Core Collection', 'biolo...",False,False
4,1728,1908.0,The Forgotten Planet,Murray Leinster,1954,1954-00-00,Ace Books,1896.0,"Norfolk, Virginia, USA",0881846163,<b>From the first page of the Ace Double:</b> ...,"['insects', 'Librivox', 'Project Gutenberg', '...",False,False
6,2876,3239.0,Rogue Queen,L. Sprague de Camp,1954,1954-00-00,Ace Books,1907.0,"New York City, New York, USA",0312943962,NaN,['science fiction'],False,False
9,5191,6101.0,The Star Beast,Robert A. Heinlein,1954,1954-08-23,Ace Books,1907.0,"Butler, Missouri, USA",0345275802,A talking alien pet has grown to the size of a...,"['aliens', 'diplomacy', 'Earth', 'first contac...",False,False
10,29917,669501.0,The Big Eye,Max Ehrlich,1954,1954-00-00,Doubleday,1909.0,"Springfield, Massachusetts, USA",NaN,NaN,"['NESFA Core Reading List', 'science fiction']",False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151586,146918,3506914.0,Daughter of the Underworld,"Katharine Corr, Elizabeth Corr",2025,2025-09-09,Candlewick Press,NaN,NaN,9781536244533.0,"Join a thrilling journey into the Underworld, ...","['Young Adult Fiction', 'Fantasy', 'Young Adul...",False,False
151587,146917,3506911.0,We Are Always Tender with Our Dead,Eric LaRocca,2025,2025-09-09,Titan Books,NaN,"Connecticut, USA",9781803368672.0,A chilling supernatural tale of transgressive ...,"['Fiction', 'LGBTQ', 'Fantasy', 'TBB The Grave...",False,False
151588,146916,3506908.0,They Fear Not Men in the Woods,Gretchen McNeil,2025,2025-09-09,DAW Books,1983.0,"San Francisco, California, USA",9780756420086.0,When Jen Monroe hears her father's remains hav...,['Fiction'],False,False
151589,146914,3506905.0,Press 1 for Invasion,J. A. Dauber,2025,2025-09-09,Aladdin,NaN,NaN,9781665974776.0,NaN,['juvenile sf'],False,False


In [162]:
books[~books['tags'].isna()][books['locus']]

/var/folders/bt/8s6v7ngs6m93f2x58bpj1zl00000gn/T/ipykernel_91165/4046754222.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  books[~books['tags'].isna()][books['locus']]


,Unnamed: 0,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,tags,hugo,locus
897,2268,2501.0,Star Gate,Andre Norton,1958,1958-00-00,"Harcourt, Brace & Company",1912.0,"Cleveland, Ohio, USA",0575040076,A story of time levels in a future world inhab...,"['science fiction', 'Fantasy', 'Young Adult', ...",False,True
2589,9153,11168.0,Space Opera,Jack Vance,1965,1965-02-00,Pyramid Books,1916.0,"San Francisco, California, USA",0934438994,NaN,"['adventure', 'aliens', 'lost colony', 'music'...",False,True
4096,6305,7649.0,Fourth Mansions,R. A. Lafferty,1969,1969-00-00,Ace Books,1914.0,"Neola, Iowa, USA",0441245900,NaN,"['biological warfare', 'fantasy', 'immigration...",False,True
4164,1921,2117.0,Deryni Rising,Katherine Kurtz,1970,1970-08-00,Ballantine Books,1944.0,"Coral Gables, Florida, USA",0345019814,NaN,['science fiction'],False,True
4232,2210,2428.0,And Chaos Died,Joanna Russ,1970,1970-00-00,Ace Books,1937.0,"Bronx, New York City, New York, USA",0441022685,NaN,"['nebula award for best novel finalist', 'scie...",False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150397,147239,3510792.0,Queen Demon,Martha Wells,2025,2025-10-07,Tor,1964.0,"Fort Worth, Texas, USA",9781250826916.0,Dahin believes he has clues to the location of...,"['fantasy', 'Fantasy', 'LGBTQ', 'Science Ficti...",False,True
150483,147381,3512193.0,The Everlasting,Alix E. Harrow,2025,2025-10-28,Tor,1989.0,"Idaho, USA",9781250799081.0,Sir Una Everlasting was Dominion’s greatest he...,"['fantasy', 'Fantasy', 'Science Fiction', 'Rom...",True,True
150886,143158,3394110.0,Death of the Author,Nnedi Okorafor,2025,2025-01-14,William Morrow & HarperAudio,1974.0,"Cincinnati, Ohio, USA",9780063391154.0,"Disabled, disinclined to marry, and more inter...","['Science Fiction', 'Young Adult', 'Adventure'...",True,True
150926,146421,3493655.0,Lessons in Magic and Disaster,Charlie Jane Anders,2025,2025-08-19,Tor,1969.0,"Tolland County, Connecticut, USA",9781250867322.0,In the vein of Alice Hoffman and Charlie Jane ...,"['Fantasy', 'Fiction', 'LGBTQ', 'Science Ficti...",False,True


In [159]:
books[books['tags'].isna()][books['locus']]

/var/folders/bt/8s6v7ngs6m93f2x58bpj1zl00000gn/T/ipykernel_91165/2662695118.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  books[books['tags'].isna()][books['locus']]


,Unnamed: 0,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,tags,hugo,locus
3867,5380,6328.0,The Palace,D. G. Compton,1969,1969-00-00,Hodder & Stoughton,1930.0,"London, England, UK",0340109688,NaN,NaN,False,True
5415,885,994.0,To Die in Italbar,Roger Zelazny,1973,1973-00-00,Methuen,1937.0,"Euclid, Ohio, USA",0413409805,<b>From the back cover of the DAW first printi...,NaN,False,True
14913,2162,2376.0,Being a Green Mother,Piers Anthony,1987,1987-12-00,Severn House,1934.0,"Oxford, Oxfordshire, England, UK",0727840010,NaN,NaN,False,True
15172,787,880.0,Way of the Pilgrim,Gordon R. Dickson,1987,1987-05-00,Ace Books,1923.0,"Edmonton, Alberta, Canada",0441874878,"<b>From the front flap of the BCE:</b> ""The Aa...",NaN,False,True
15221,1467,1626.0,Little Heroes,Norman Spinrad,1987,1987-07-00,Bantam Spectra,1940.0,"New York City, New York, USA",0553270338,NaN,NaN,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144709,142757,3376479.0,Blood,Sarah Pinborough,2024,2024-11-28,Gollancz,1972.0,"Stony Stratford, Buckinghamshire, England, UK",9781399623452.0,NaN,NaN,False,True
148719,143483,3403712.0,Aurelia,Stephen R. Lawhead,2025,2025-01-07,Baen Books,1950.0,"Good Samaritan Hospital, Kearney, Nebraska, USA",9781625799982.0,NaN,NaN,False,True
148872,145414,3466220.0,The Folded Sky,Elizabeth Bear,2025,2025-06-17,Saga Press,1971.0,"Hartford, Connecticut, USA",9781668078112.0,NaN,NaN,False,True
150353,147292,3511083.0,All That We See or Seem,Ken Liu,2025,2025-10-09,Ad Astra / Head of Zeus,1976.0,"Lanzhou, Gansu Province, China",9781035915941.0,NaN,NaN,False,True


In [150]:
nonfinalists['first_publisher'].value_counts(normalize=True)#.reset_index(name='counts').set_index('first_publisher').plot.pie(y='counts', legend=False)

first_publisher
ace                  0.068244
doubleday            0.028787
roberthale           0.023452
ballantinebooks      0.023352
bantambooks          0.017413
                       ...   
voidpublications     0.000101
northpointpress      0.000101
whitedove            0.000101
f.a.thorpe           0.000101
thepermanentpress    0.000101
Name: proportion, Length: 1262, dtype: float64

In [154]:
books_eda['first_publisher'].value_counts(normalize=True)

first_publisher
ace                  0.069073
doubleday            0.029268
ballantinebooks      0.024293
roberthale           0.022829
bantambooks          0.017659
                       ...   
voidpublications     0.000098
northpointpress      0.000098
whitedove            0.000098
f.a.thorpe           0.000098
thepermanentpress    0.000098
Name: proportion, Length: 1271, dtype: float64

In [ ]:
finalists_exploded = finalists.explode('first_publisher')

In [127]:
publisher_performance = finalists_exploded.groupby(['release_year', 'first_publisher'])['hugo'].mean().unstack()

In [131]:
print(publisher_performance['ace'])

release_year
1954    NaN
1955    NaN
1956    NaN
1958    0.0
1959    NaN
1960    1.0
1961    NaN
1962    1.0
1963    1.0
1964    NaN
1965    NaN
1966    1.0
1967    1.0
1968    1.0
1969    0.5
1970    0.2
1971    0.0
1972    0.0
1973    NaN
1974    1.0
1975    NaN
1976    NaN
1977    0.0
1978    0.0
1979    0.5
1980    1.0
Name: ace, dtype: float64


In [132]:
import seaborn as sns

sns.boxplot(
    data=finalists_exploded['ace'], 
    x='first_publisher', 
    y='hugo', 
    hue='release_year', 
    palette='Set2'
)

KeyError: 'ace'

### Authors

In [45]:
finalists['author'].value_counts()

author
Roger Zelazny         13
Robert Silverberg     13
Poul Anderson         12
Robert A. Heinlein     9
Jack Vance             7
                      ..
Peter Straub           1
Keith Roberts          1
Lyndon Hardy           1
Basil Copper           1
Robert Stallman        1
Name: count, Length: 135, dtype: int64

In [46]:
nonfinalists['author'].value_counts()

author
Andre Norton                      76
Edgar Rice Burroughs              64
Dorothy Daniels                   59
Ron Goulart                       46
Lin Carter                        45
                                  ..
David A. Kyle                      1
Rudy Rucker                        1
Gregory Benford, Gordon Eklund     1
Ernest T. Jahn                     1
Robert P. Davis                    1
Name: count, Length: 4269, dtype: int64

### Tags cleaning

In [793]:
books['tags'][100]

nan

In [794]:
from pattern.en import singularize

In [831]:
def flatten_clean_tags(taglist):
    if isinstance(taglist, float):
        return taglist
    else:
        cleaned = [item.translate(str.maketrans('', '', string.punctuation)).lower().strip(' ') for item in taglist.strip('[]').replace('  ', ',').split(',')]
        cleaned = [stemmer.stem(singularize(word)) for word in cleaned]
        cleaned = [lemmatizer.lemmatize(word) for word in cleaned]
        return cleaned
books['tags'] = books['tags'].apply(flatten_clean_tags)


In [832]:
tag_counts = books['tags'].explode().value_counts()

illegal = ['locus', 'hugo', 'winner', 'award', 'best', 'tv', 'movie', "ofearnaaudible", "ofearnaaudio", 'audiobook', 'audible', "ofearnaebook", "ofearnaenook", 'audiobook', 'ofearnaaud', 'ofearnaread', 'intovideo gam', 'movi', 'ofearnanovel', 'ofearnacov', 'classic', 'classiqu', 'read', 'toread', 'big sky librari', 'big sky library  to find', 'comics  graphic novel', 'graphic novel', 'literary critic', 'literary collect', 'fromaud', 'fromna', 'intomus', 'intocom', 'author', 'male author',
           
    "project gutenberg",
    "librivox",
    "internet arch",
    "audio book",
    "audibl",
    "google book",
    "googleplay",
    "storybundl",
    "humblebundl",
    "kindl",
    "electronic book",
    "manybook",
    "large type book",
    "ofearnaseriesnovel",
    "ofearnakindl",
    "ofearnanook",
    "appendix e of player’s handbook",
    "appendix n",
    "recursive sf nesfa index",
    "gollancz sf masterwork",
    "scifimasterwork",
    "possible nongenr",
    "todo",
    "pick",
    "add",
    "wishlist",
    "all",
    "gener",
    "novel",
    "fictiun",
    "genre fict",
    "fiction in english"
]


valid_tags = tag_counts[(tag_counts >= 20 )& (~tag_counts.index.str.contains("|".join(illegal), case=False)) & (~tag_counts.index.str.contains(r"\d", na=False)) & (tag_counts.index != "fiction")].index.tolist()

In [833]:
print(len(valid_tags))

568


In [834]:
valid_tags

['fantasi',
 'science fict',
 'adventur',
 'young adult',
 'youngadult fantasi',
 'juvenile fantasi',
 'paranormal rom',
 '',
 'mysteri',
 'horror',
 'urban fantasi',
 'dystopian',
 'alien',
 'loveable charact',
 'strong character develop',
 'dark',
 'a mix driven',
 'diverse charact',
 'vampir',
 'emot',
 'ten',
 'romanc',
 'weak character develop',
 'space',
 'youngadult sf',
 'unloveable charact',
 'lgbtq',
 'not diverse charact',
 'war',
 'plot driven',
 'time travel',
 'historical fantasi',
 'character driven',
 'funni',
 'young adult fict',
 'thriller',
 'juvenile sf',
 'reflect',
 'sad',
 'post apocalyps',
 'shapechang',
 'hope',
 'space opu',
 'near futur',
 'juvenile fict',
 'magic',
 'dystopium',
 'ghost',
 'werewolf',
 'lightheart',
 'military sf',
 'zombi',
 'youngadult horror',
 'inspir',
 'alternate histori',
 'suspens',
 'histori',
 'steampunk',
 'dragon',
 'fastpac',
 'witch',
 'paranormal mysteri',
 'humor',
 'science fiction  fantasi',
 'superhero',
 'mediumpac',
 'co

In [835]:
flatten_clean_tags("[historical]")

['histor']

In [768]:
'classic', 'classiqu', 'read', 'toread', 'big sky librari', 'big sky library  to find', 'comics  graphic novel', 'graphic novel', 'literary critic', 'literary collect', 'fromaud', 'fromna'


('classic',
 'classiqu',
 'read',
 'toread',
 'big sky librari',
 'big sky library  to find',
 'comics  graphic novel',
 'graphic novel',
 'literary critic',
 'literary collect',
 'fromaud',
 'fromna')

In [769]:
from rapidfuzz import fuzz, process

similar = []

for i in range(len(valid_tags)):
    fuzzymatching = process.extract(
        valid_tags[i],
        valid_tags[i+1:],
        scorer = fuzz.ratio, 
        score_cutoff=0.7
    )

    for match, score, index in fuzzymatching:
        similar.append((valid_tags[i], match, score))

In [770]:
i = 0
for str1, str2, score in similar:
    ignore = ['fantasi', 'adventur', 'horror', 'urban fantasi', 'paranormal rom', 'dystopian', 'loveable charact', 'diverse charact', ['near futur']]
    if float(score) > 70 and score <= 75 not in ignore:
        print(f"'{str2}' : '{str1}'")
        print(i)
        i += 1

'epic fantasi' : 'fantasi'
0
'dark fantasi' : 'fantasi'
1
'cozy fantasi' : 'fantasi'
2
'animal fantasi' : 'urban fantasi'
3
'christian fantasi' : 'urban fantasi'
4
'dark fantasi' : 'urban fantasi'
5
'monster' : 'mysteri'
6
'diverse charact' : 'loveable charact'
7
'empir' : 'vampir'
8
'erot' : 'emot'
9
'tiein' : 'ten'
10
'not diverse charact' : 'unloveable charact'
11
'dwarf' : 'war'
12
'water' : 'war'
13
'travel' : 'time travel'
14
'magical r' : 'magic'
15
'fantasy litrpg' : 'fantasy rom'
16
'cozy fantasi' : 'epic fantasi'
17
'military fantasi' : 'dark fantasi'
18
'arthurian fantasi' : 'animal fantasi'
19
'rebellion' : 'religion'
20
'teleport' : 'telepathi'
21
'mytholog' : 'greek mytholog'
22
'africa' : 'american'
23
'literari' : 'militari'
24
'rat' : 'pirat'
25
'fairi' : 'famili'
26
'ireland' : 'england'
27
'longev' : 'clone'
28
'coloni' : 'clone'
29
'weird' : 'wizard'
30
'star war' : 'star trek'
31
'humorous fantasi' : 'religious fantasi'
32
'norse mytholog' : 'mytholog'
33
'psycholo

In [771]:
flatten_clean_tags("['suspense]")

['suspens']

In [ ]:
canonical_tag_matching = {
    "science fict": "sf",
    "scienc fict": "sf",
    "sciense fict": "sf",
    "young adult": "ya",
    "youngadult": "ya",
    "teen  young adult": "ya",
    "aventur": "adventur",
    "aventure": "adventur",
    "adventurers": "adventur",
    "aventura": "adventur",
    "adventurous": "adventur",
    "adventure plot": "adventur",
    "misadventures": "adventur",
    "adventures": "adventur",
    "young adult fantasi": ["ya", "fantasi"],
    "youngadultfantasi": ["ya", "fantasi"],
    "youngadult fantasi": ["ya", "fantasi"],
    "teen fantasi": ["ya", "fantasi"],
    "juvenile fantasi": ["ya", "fantasi"],
    "teen urban fantasi": ["ya", "urban fantasi"],
    "juvenile urban fantasi": ["ya", "urban fantasi"],
    "paranormal erotic romance": "paranormal rom",
    "paranormal romance stories": "paranormal rom",
    "paranormal  urban": "paranorm",
    "paranormal powers": "paranorm",
    "paranormal mystery": "paranorm",
    "horreur": "horror",
    "read horror": "horror",
    "body horror": "horror",
    "body horror": "horror",
    "dystopia": "dystopian",
    "dystopia": "dystopias",
    "alien" : "aliens",
    "military sf": ["militari", "sf"],
    "juvenile sf": ["ya", "sf"],
    "suspenseful": "suspense",
    "strong female characters": "strong female character",
    "humor": "humour",
    "literature  fiction": "literari",
    "romanc": "rom",
    "romant": "rom",
    "youngadult sf": ["ya", "sf"],
    'youngadult fict': "ya",
    "lgbt" : "lgbtq",
    "time travel rom": ['rom', 'time travel'],
    'teen historical fantasi': ['teen', 'historical fantasi'],
    "juvenile fict": "ya",
    "post apocalypt": "apocalyps",
    "postapocalypt": "apocalyps",
    "historical fantasy rom": ['historical fantasi', 'fantasy rom'],
    "youngadult rom": ['ya', 'rom'],
    "youngadult horror": ['ya', 'horror'],
    "histor": 'histori',
    'dragonl': 'dragon',
    'paranormal mysteri': 'paranormal',
    'paranormal thril': 'paranormal',
    'science fiction  fantasi': ['sf', 'fantasi'],
    'science fiction fantasi': ['sf', 'fantasi'],
    'psychic ': 'psychic',
    'fantasyrom': 'fantasy rom',
    'nanotechnolog' : 'technolog',
    'literature  fict' : 'literari',
    'literatur' : 'literari',
    'action  adventur' : 'adventur',
    'actionadventur' : 'adventur',
    'apocalypt' : 'apocalyps',
    'youngadult historical fantasi': ['ya', 'historical fantasi'],
    'youngadult animal fantasi': ['ya', 'animal fantasi'],
    'young adult urban fantasi': ['ya', 'urban fantasi'],
    'mad scientist': 'scientist',
    'christian sf' : ['christian', 'sf'],
    'christian fict': 'christian',
    'colon' : 'coloni',
    'archeolog' : 'archaeolog',
    'paranormal erotic rom' : 'paranormal rom',
    'paranormal fict' : 'paranormal',
    'dystopium' : 'dystopian',
    'space st': 'space',
    'historical fict': 'histori',
    'timetravel rom': ['time travel', 'rom'],
    'post apocalyps' : 'apocalyps',
    'magician' : 'magic',
    'alternative histories fict': 'alternate histori',
    'literary fict': 'literari',
    'fantasy fict' : 'fantasi',
    'gay erotica': ['lgbtq', 'erot'],
    'american fict': 'american',
    'teen science fict': ['ya', 'sf'],
    'faeri' : 'fairi',
    'african speculative fict': 'african',
    'literary fict': 'literari',
    'literary fantasi': ['literari', 'fantasi'],
    'interplanetary travel' : 'interplanetary voyag',
    'romantic fantasi': 'fantasy rom',
    'sf romanc': 'sf rom',
    'mm romanc': 'lgbtq',
    'paranormal fict' : 'paranormal',
    'paranormal pow' : 'paranormal',
    'dragons  mythical creatur': ['dragon', 'mythical'], 
    'romantasi' : 'fantasy rom',
    'fantasyhummi' : 'fantasi',
    'paranormal romance stori' : 'paranormal rom', 
    'paranormal' : 'paranorm',
    'horror tal' : 'horror', 
    'cozy mysteri' : ['mysteri', 'cozy'],
    'vampire rom' : ['vampir', 'rom'],
    'space opu' : 'space',
    'youngadult vampir' : ['ya', 'vampir'],
    'young adult fict': 'ya',
    'youngadult timetravel' : ['ya', 'time travel'],
    'prehistor' : 'histori',
    'alternate world' : 'alternate histori',
    'secret histori' : 'alternate histori',
    'alternate univers' : 'alternate histori',
    'superpow' : 'superhero',
    'alternate univers' : 'alternate histori',
    'demonolog' : 'demon',
    'futuristic mysteri' : ['futuristic', 'mysteri'],
    'futuristic rom': ['futuristic', 'rom'],
    'erotica' : 'erot',
    'supernatural thril' : 'supernatur',
    'contemporary fantasi' : ['contemporari', 'fantasy'],
    'supernatural thril' : 'supernatur',
    'retold fairy tal': 'fairi',
    'english fict' : 'english',
    'scifi rom' : 'sf rom',
    'paranormal suspens' : ['paranorm', 'suspens'],
    'thriller  suspensescience fiction  fantasi': ['thriller', 'sf', 'suspens', 'fantasi'],
    'hard science fict' : 'hard sf',
    'lost coloni' : 'coloni',
    'juvenil': 'ya',
    'future australium' : 'australium',


    # --- Fantasy Subgenres & Lore ---
    "fantezi": ["fantasi"],
    "epic": ["epic fantasi"],
    "high fantasi": ["epic fantasi"],
    "heroic fantasi": ["epic fantasi"],
    "quest": ["fantasi"],
    "progression fantasi": ["fantasi"],
    "fairy tale inspir": ["fairy tal"],
    "retel": ["fairy tal"],
    "cinderella": ["fairy tal"],
    "the sleeping beauti": ["fairy tal"],
    "beauty and the beast": ["fairy tal"],

    # --- Sci-Fi Subgenres & Space Tropes ---
    "sf": ["sf"],
    "printsf": ["sf"],
    "hard sf": ["hard sf"],
    "british dystopium": ["dystopian", 'cozy'],
    "utopium": ["utopium"],
    "end of the world": ["apocalyps"],
    "cosy catastroph": ["apocalyps"],
    "hitler win": ["alternate histori"],
    "star trek the original seri": ["star trek"],
    "interstellar travel": ["space travel"],
    "interplanetary voyag": ["space travel"],
    "space warfar": ["militari", 'space'],
    "space coloni": ["space travel"],
    "galactic empir": ["space travel"],
    "life on other planet": ["space travel"],
    "alien invas": ["alien"],
    "first contact": ["alien"],
    "alien artifact": ["alien"],
    "humanalien encount": ["alien"],
    "war with alien": ["alien"],
    "alien perspect": ["alien"],
    "martian": ["alien"],
    "extraterrestrial b": ["alien"],

    # --- Cyberpunk, LitRPG & Tech ---
    "computer gam": ["video gam"],
    "gamebook": ["video gam"],
    "transhuman": ["cyberpunk"],
    "posthuman": ["cyberpunk"],

    # --- Horror & Paranormal Subgenres ---
    "juvenile horror": ["ya", "horror"],
    "gothic horror": ["gothic"],
    "scari": ["horror"],
    "terror": ["horror"],
    "lovecraftian": ["horror"],
    "bizarro": ["weird"],
    "haunted hous": ["ghost"],
    "ghost stori": ["ghost"],
    "youngadult ghost stori": ["ya", "ghost"],

    # --- Paranormal Beings & Occult ---
    "shapeshift": ["shapechang"],
    "shifter": ["shapechang"],
    "lycanthropi": ["werewolf"],
    "dragon shift": ["shapechang", "dragon"],
    "youngadult supernatur": ["ya", "paranorm"],
    "occult detect": ["paranorm"],

    # --- Romance Categories (Separated) ---
    "fantasy rom": ["fantasy rom"],
    "monster rom": ["monster rom"],
    "gay rom": ["lgbtq", "rom"],
    "rom": ["rom"],
    "love stori": ["rom"],
    "love": ["rom"],
    "fated m": ["rom"],
    "manwoman relationship": ["rom"],
    "erotic stori": ["erot"],

    # --- Psychic & Biological Tropes ---
    "telepathi": ["psy pow"],
    "precognit" : ["psy pow"],
    "psychic": ["psy pow"],
    "clairvoy": ["psy pow"],
    "esp": ["psy pow"],
    "telekinesi": ["psy pow"],
    "clone": ["genetic engin"],
    "mutant": ["genetic engin"],

    # --- Pacing, Narrative Mechanics & POV ---
    "first person point of view": ["first person"],
    "thriller  suspens": ["thriller", "suspens"],
    "detective and mystery stori": ["mysteri"],
    "murder mysteri": ["mysteri"],
    "adventure stori": ["adventur"],
    "adventure and adventur": ["adventur"],
    "fantastic adventur": ["adventur", "fantasi"],
    "planetary adventur": ["adventur", 'fantasi'],
    "sea adventur": ["adventur"],

    # --- Splits & Intersections (Age Groups) ---
    "teenag": ["ya"],
    "child": ["ya"],
    "middle grades fantasi": ["ya", "fantasi"],
    "middle grades science fict": ["ya", "sf"],
    "juvenile mysteri": ["ya", "mysteri"],

    # --- Real-World Settings mapped to core regional stems ---
    "united st": ["american"],
    "new york c": ["american"],
    "californium": ["american"],
    "san francisco": ["american"],
    "new orlean": ["american"],
    "london": ["england"],
    "great britain": ["england"],
    "uk": ["england"],
    "scotland": ["england"],
    "ireland": ["england"]
}


ml_genre_reduction = {
    # --- Sci-Fi & Space Realignment ---
    "sf": ["scifi"],
    "printsf": ["scifi"],
    "hard sf": ["scifi"],
    "science fantasi": ["scifi", "fantasi"],
    "interstellar travel": ["space travel"],
    "interplanetary voyag": ["space travel"],
    "space coloni": ["space travel"],
    "generation ship": ["space travel"],
    "space warfar": ["space travel", "militari"],
    "space pir": ["space travel", "adventur"],
    "alien invas": ["alien"],
    "first contact": ["alien"],
    "alien artifact": ["alien"],
    "humanalien encount": ["alien"],
    "war with alien": ["alien"],
    "alien perspect": ["alien"],
    "martian": ["alien"],
    "extraterrestrial b": ["alien"],
    "star trek the original seri": ["star trek"],

    # --- Fantasy Realignment ---
    "high fantasi": ["epic fantasi"],
    "heroic fantasi": ["epic fantasi"],
    "sword and sorceri": ["epic fantasi"],
    "quest": ["epic fantasi"],
    "progression fantasi": ["epic fantasi"],
    "fantezi": ["fantasi"],
    "fairy tale inspir": ["fairy tal"],
    "folk tal": ["fairy tal"],
    "cinderella": ["fairy tal"],
    "the sleeping beauti": ["fairy tal"],
    "beauty and the beast": ["fairy tal"],

    # --- Reality & Timeline Shifts ---
    "end of the world": ["apocalyps"],
    "cosy catastroph": ["apocalyps"],
    "parallel world": ["parallel univers"],
    "transdimension": ["parallel univers"],
    "portalg": ["parallel univers"],
    "time loop": ["time travel"],
    "time slip": ["time travel"],
    "juvenile timetravel": ["time travel", "ya"],
    "hitler win": ["alternate histori"],
    "lost world": ["lost rac"],

    # --- Paranormal & Magic Consolidations ---
    "paranorm": ["supernatur"],
    "paranormal": ["supernatur"],
    "occult": ["supernatur"],
    "occult detect": ["supernatur", "mysteri"],
    "shapeshift": ["shapechang"],
    "shifter": ["shapechang"],
    "lycanthropi": ["werewolf"],
    "dragon shift": ["shapechang", "dragon"],
    "haunted hous": ["ghost"],
    "ghost stori": ["ghost"],

    # --- Romance Categorization ---
    "rom": ["paranormal rom"], # Minor count fallback
    "love stori": ["paranormal rom"],
    "love": ["paranormal rom"],
    "fated m": ["paranormal rom"],
    "manwoman relationship": ["paranormal rom"],
    "erotic stori": ["erot"],
    "steami": ["erot"],

    # --- Plot, Pacing & Mystery ---
    "thriller  suspens": ["thriller"],
    "suspens": ["thriller"],
    "detective and mystery stori": ["mysteri"],
    "murder mysteri": ["mysteri"],
    "private investig": ["mysteri"],
    "detect": ["mysteri"],
    "adventure stori": ["adventur"],
    "adventure and adventur": ["adventur"],
    "fantastic adventur": ["adventur"],
    "planetary adventur": ["adventur"],
    "sea adventur": ["adventur"],

    # --- Demographics & Representation ---
    "female protagonist": ["female lead"],
    "female main charact": ["female lead"],
    "strong female charact": ["female lead"],
    "gay": ["lgbtq"],
    "lesbian": ["lgbtq"],
    "queer": ["lgbtq"],
    "bisexu": ["lgbtq"],
    "transgend": ["lgbtq"],
    "nonbinary gend": ["lgbtq"],
    "sapphic": ["lgbtq"],
    "gay rom": ["lgbtq", "paranormal rom"],

    # --- Tech, Gaming & Cyber-Tropes ---
    "computer gam": ["video gam"],
    "gamebook": ["video gam"],
    "transhuman": ["cyberpunk"],
    "posthuman": ["cyberpunk"],
    "android": ["robot"],
    "cyborg": ["robot"],
    "artificial intellig": ["ai"],
}


In [818]:
tag_mapping = {
    "science fict": "sf",
    "scienc fict": "sf",
    "sciense fict": "sf",
    "young adult": "ya",
    "youngadult": "ya",
    "teen  young adult": "ya",
    "aventur": "adventur",
    "aventure": "adventur",
    "adventurers": "adventur",
    "aventura": "adventur",
    "adventurous": "adventur",
    "adventure plot": "adventur",
    "misadventures": "adventur",
    "adventures": "adventur",
    "young adult fantasi": ["ya", "fantasi"],
    "youngadultfantasi": ["ya", "fantasi"],
    "youngadult fantasi": ["ya", "fantasi"],
    "teen fantasi": ["ya", "fantasi"],
    "juvenile fantasi": ["ya", "fantasi"],
    "teen urban fantasi": ["ya", "urban fantasi"],
    "juvenile urban fantasi": ["ya", "urban fantasi"],
    "paranormal erotic romance": "paranormal rom",
    "paranormal romance stories": "paranormal rom",
    "paranormal  urban": "paranorm",
    "paranormal powers": "paranorm",
    "paranormal mystery": "paranorm",
    "horreur": "horror",
    "read horror": "horror",
    "body horror": "horror",
    "body horror": "horror",
    "dystopia": "dystopian",
    "dystopia": "dystopias",
    "alien" : "aliens",
    "military sf": ["militari", "sf"],
    "juvenile sf": ["ya", "sf"],
    "suspenseful": "suspense",
    "strong female characters": "strong female character",
    "humor": "humour",
    "literature  fiction": "literari",
    "romanc": "rom",
    "romant": "rom",
    "youngadult sf": ["ya", "sf"],
    'youngadult fict': "ya",
    "lgbt" : "lgbtq",
    "time travel rom": ['rom', 'time travel'],
    'teen historical fantasi': ['teen', 'historical fantasi'],
    "juvenile fict": "ya",
    "post apocalypt": "apocalyps",
    "postapocalypt": "apocalyps",
    "historical fantasy rom": ['historical fantasi', 'fantasy rom'],
    "youngadult rom": ['ya', 'rom'],
    "youngadult horror": ['ya', 'horror'],
    "histor": 'histori',
    'dragonl': 'dragon',
    'paranormal mysteri': 'paranormal',
    'paranormal thril': 'paranormal',
    'science fiction  fantasi': ['sf', 'fantasi'],
    'science fiction fantasi': ['sf', 'fantasi'],
    'psychic ': 'psychic',
    'fantasyrom': 'fantasy rom',
    'nanotechnolog' : 'technolog',
    'literature  fict' : 'literari',
    'literatur' : 'literari',
    'action  adventur' : 'adventur',
    'actionadventur' : 'adventur',
    'apocalypt' : 'apocalyps',
    'youngadult historical fantasi': ['ya', 'historical fantasi'],
    'youngadult animal fantasi': ['ya', 'animal fantasi'],
    'young adult urban fantasi': ['ya', 'urban fantasi'],
    'mad scientist': 'scientist',
    'christian sf' : ['christian', 'sf'],
    'christian fict': 'christian',
    'colon' : 'coloni',
    'archeolog' : 'archaeolog',
    'paranormal erotic rom' : 'paranormal rom',
    'paranormal fict' : 'paranormal',
    'dystopium' : 'dystopian',
    'space st': 'space',
    'historical fict': 'histori',
    'timetravel rom': ['time travel', 'rom'],
    'post apocalyps' : 'apocalyps',
    'magician' : 'magic',
    'alternative histories fict': 'alternate histori',
    'literary fict': 'literari',
    'fantasy fict' : 'fantasi',
    'gay erotica': ['lgbtq', 'erot'],
    'american fict': 'american',
    'teen science fict': ['ya', 'sf'],
    'faeri' : 'fairi',
    'african speculative fict': 'african',
    'literary fict': 'literari',
    'literary fantasi': ['literari', 'fantasi'],
    'interplanetary travel' : 'interplanetary voyag',
    'romantic fantasi': 'fantasy rom',
    'sf romanc': 'sf rom',
    'mm romanc': 'lgbtq',
    'paranormal fict' : 'paranormal',
    'paranormal pow' : 'paranormal',
    'dragons  mythical creatur': ['dragon', 'mythical'], 
    'romantasi' : 'fantasy rom',
    'fantasyhummi' : 'fantasi',
    'paranormal romance stori' : 'paranormal rom', 
    'paranormal' : 'paranorm',
    'horror tal' : 'horror', 
    'cozy mysteri' : ['mysteri', 'cozy'],
    'vampire rom' : ['vampir', 'rom'],
    'space opu' : 'space',
    'youngadult vampir' : ['ya', 'vampir'],
    'young adult fict': 'ya',
    'youngadult timetravel' : ['ya', 'time travel'],
    'prehistor' : 'histori',
    'alternate world' : 'alternate histori',
    'secret histori' : 'alternate histori',
    'alternate univers' : 'alternate histori',
    'superpow' : 'superhero',
    'alternate univers' : 'alternate histori',
    'demonolog' : 'demon',
    'futuristic mysteri' : ['futuristic', 'mysteri'],
    'futuristic rom': ['futuristic', 'rom'],
    'erotica' : 'erot',
    'supernatural thril' : 'supernatur',
    'contemporary fantasi' : ['contemporari', 'fantasy'],
    'supernatural thril' : 'supernatur',
    'retold fairy tal': 'fairi',
    'english fict' : 'english',
    'scifi rom' : 'sf rom',
    'paranormal suspens' : ['paranorm', 'suspens'],
    'thriller  suspensescience fiction  fantasi': ['thriller', 'sf', 'suspens', 'fantasi'],
    'hard science fict' : 'hard sf',
    'lost coloni' : 'coloni',
    'juvenil': 'ya',
    'future australium' : 'australium',

    # --- Exact Text Matches / Un-Stemmed Plural Cleanups ---
    "fantasy": ["fantasi"],
    "aliens": ["alien"],
    "dystopias": ["dystopian"],
    "british dystopium": ["dystopian"],
    "humour": ["funni"],
    "sf rom": ["scifi", "rom"],
    "dark rom": ["dark", "rom"],
    "monster rom": ["monster rom"],
    "fantasy rom": ["fantasy rom"],
    "paranormal": ["supernatur"],
    "paranorm": ["supernatur"],
    "utopium": ["dystopian"],  # Collapsed to its core societal pole
    "first person point of view": ["first person"],
    "sword and sorceri": ["epic fantasi"],
    "sword  sorceri": ["epic fantasi"],

    # --- Specific Magic Beings & High Fantasy Elements ---
    "fairi": ["fantasi"],
    "wizard": ["fantasi"],
    "propheci": ["fantasi"],
    "secret": ["fantasi"],
    "retel": ["fairy tal"],
    "mythical": ["fairy tal"],
    "discworld": ["fantasi"],
    "arthurian fantasi": ["historical fantasi"],
    "arthurian rom": ["historical fantasi", "rom"],
    "cozy": ["lightheart"],
    "cozi": ["lightheart"],

    # --- Hard Sci-Fi, Deep Space & Tech Elements ---
    "hard sf": ["scifi"],
    "speculative fict": ["scifi"],
    "futuristic": ["near futur"],
    "far futur": ["near futur"],
    "distant futur": ["near futur"],
    "technolog": ["scifi"],
    "robot": ["robot"],
    "angry robot": ["robot"],
    "ai": ["ai"],
    "enhanced intellig": ["ai"],
    "higher intellig": ["ai"],
    "video gam": ["video gam"],
    "litrpg": ["litrpg"],
    "cyberpunk": ["cyberpunk"],
    "steampunk": ["steampunk"],
    "airship": ["steampunk"],
    "life on other planet": ["space travel"],
    "interplanetary voyag": ["space travel"],
    "galactic empir": ["space travel"],
    "coloni": ["space travel"],
    "star trek": ["space travel"],
    "climate chang": ["apocalyps"],
    "ecolog": ["apocalyps"],

    # --- Mystery, Thriller, Action & Intrigue ---
    "suspens": ["thriller"],
    "action": ["adventur"],
    "murder": ["mysteri"],
    "private investig": ["mysteri"],
    "detect": ["mysteri"],
    "crime": ["mysteri"],
    "juvenile mysteri": ["mysteri"],
    "spy": ["mysteri"],
    "spi": ["mysteri"],
    "espionag": ["mysteri"],
    "assassin": ["mysteri"],
    "conspiraci": ["thriller"],
    "intrigu": ["thriller"],
    "kidnap": ["thriller"],
    "serial kil": ["thriller"],
    "missing person": ["mysteri"],
    "abduct": ["thriller"],
    "pirat": ["adventur"],
    "escap": ["adventur"],
    "rescu": ["adventur"],
    "surviv": ["adventur"],
    "explor": ["adventur"],
    "travel": ["adventur"],
    "faster than light travel": ["space travel"],
    "martial art": ["adventur"],

    # --- Pacing and Perspective Formats ---
    "fastpac": ["fastpac"],
    "mediumpac": ["mediumpac"],
    "slowpac": ["slowpac"],
    "slow burn": ["slowpac"],
    "first person": ["first person"],
    "multiple points of view": ["multiple points of view"],

    # --- Biological & Science Elements ---
    "genetic engin": ["genetic engin"],
    "clone": ["genetic engin"],
    "mutant": ["genetic engin"],
    "evolut": ["genetic engin"],
    "cryogen": ["genetic engin"],
    "suspended anim": ["genetic engin"],
    "scientist": ["genetic engin"],
    "scienc": ["genetic engin"],
    "drug": ["genetic engin"],

    # --- Demographics, Representation & Age ---
    "female lead": ["female lead"],
    "female protagonist": ["female lead"],
    "female main charact": ["female lead"],
    "strong female charact": ["female lead"],
    "female warrior": ["female lead"],
    "woman": ["female lead"],
    "written by woman": ["female lead"],
    "gay": ["lgbtq"],
    "lesbian": ["lgbtq"],
    "queer": ["lgbtq"],
    "bisexu": ["lgbtq"],
    "transgend": ["lgbtq"],
    "nonbinary gend": ["lgbtq"],
    "sapphic": ["lgbtq"],
    "gay rom": ["lgbtq", "rom"],
    "teenag": ["ya"],
    "teen": ["ya"],
    "boarding school": ["ya"],
    "magic school": ["fantasi"],
    "middle grades fantasi": ["ya", "fantasi"],
    "middle grades science fict": ["ya", "scifi"],
    "child": ["childrens stori"],
    "childrens stori": ["childrens stori"],
    "adult": ["adult"],
    "na driven": ["adult"],

    # --- Settings, Regions & Cultural History ---
    "united st": ["american"],
    "new york c": ["american"],
    "californium": ["american"],
    "san francisco": ["american"],
    "new orlean": ["american"],
    "native american": ["american"],
    "african american": ["american"],
    "london": ["england"],
    "great britain": ["england"],
    "uk": ["england"],
    "scotland": ["england"],
    "ireland": ["england"],
    "english": ["england"],
    "franc": ["contemporari"],
    "china": ["contemporari"],
    "japan": ["contemporari"],
    "egypt": ["contemporari"],
    "africa": ["contemporari"],
    "south africa": ["contemporari"],
    "mexico": ["contemporari"],
    "canada": ["contemporari"],
    "canadian fict": ["contemporari"],
    "thailand": ["contemporari"],
    "antarctica": ["contemporari"],
    "australium": ["contemporari"],
    "indium": ["contemporari"],
    "spanish": ["contemporari"],
    "african": ["contemporari"],
    "western": ["histori"],
    "weird west": ["histori"],
    "victorian": ["histori"],
    "noir": ["histori"],
    "cold war": ["histori"],
    "civil war": ["histori"],
    "world war iu": ["histori"],
    "archaeolog": ["histori"],
    "militari": ["war"],
    "military fantasi": ["war"],
    "imaginary wars and battl": ["war"],
    "interstellar war": ["war"],
    "revolut": ["war"],
    "rebellion": ["war"],
    "polit": ["war"],
    "empir": ["war"],

    # --- Narrative Tone & Themes ---
    "satir": ["funni"],
    "parodi": ["funni"],
    "comic": ["funni"],
    "hummi": ["funni"],
    "insan": ["sad"],
    "suicid": ["sad"],
    "death": ["sad"],
    "reveng": ["sad"],
    "betray": ["sad"],
    "cold": ["sad"],
    "good and evil": ["challeng"],
    "social commentari": ["challeng"],
    "social critic": ["challeng"],
    "social cla": ["challeng"],
    "racism": ["challeng"],
    "slaveri": ["challeng"],
    "genocid": ["challeng"],
    "surreal": ["reflect"],
    "philosophi": ["reflect"],
    "psycholog": ["reflect"],
    "literari": ["reflect"],
    "coming of ag": ["friendship"],
    "famili": ["friendship"],
    "sibl": ["friendship"],
    "brother": ["friendship"],
    "brothers and sist": ["friendship"],
    "twin": ["friendship"],
    "orphan": ["friendship"],
    "friendship": ["friendship"],
    "religion": ["religion"],
    "christian": ["religion"],

    # --- Animals & Mythical Creatures ---
    "cat": ["anim"],
    "dog": ["anim"],
    "hors": ["anim"],
    "rat": ["anim"],
    "mous": ["anim"],
    "spider": ["anim"],
    "dolphin": ["anim"],
    "dinosaur": ["anim"],
    "insect": ["anim"],
    "talking anim": ["anim", "fairy tal"],
    "anthropomorph": ["anim"],
    "selki": ["anim"],
    "animal fantasi": ["fantasi", "anim"],

    # --- Locations, Geography & Settings ---
    "forest": ["imaginary plac"],
    "desert": ["imaginary plac"],
    "water": ["imaginary plac"],
    "sea": ["imaginary plac"],
    "small town": ["imaginary plac"],
    "circu": ["imaginary plac"],
    "librari": ["imaginary plac"],
    "hollow earth": ["imaginary plac"],
    "basebal": ["imaginary plac"],
    "sport": ["imaginary plac"],

    # --- Arts & Media ---
    "music": ["art"],
    "poetri": ["art"],
    "food": ["art"],
    "art": ["art"],

    # --- Meta & Structural Elements ---
    "metafict": ["reflect"],
    "choose your own adventur": ["adventur"],
    "fix up": ["adventur"],
    "standalon": ["adventur"],
    "book": ["adventur"],
    "short stori": ["adventur"],

    # --- Explicit Catch-alls for Items Not Covered in Drops ---
    "cur": ["challeng"],
    "wish": ["hope"],
    "dream": ["reflect"],
    "fate": ["reflect"],
    "invent": ["genetic engin"],
    "game": ["video gam"],
    "alcohol": ["sad"],
    "sherlock holm": ["mysteri"],


    # --- Fantasy Pillars & Magic ---
    "fantezi": ["fantasi"],
    "epic": ["epic fantasi"],
    "high fantasi": ["epic fantasi"],
    "heroic fantasi": ["epic fantasi"],
    "quest": ["epic fantasi"],
    "progression fantasi": ["epic fantasi"],
    "magic": ["magic"],
    "wizard": ["magic"],
    "witch": ["magic"],
    "druid": ["magic"],
    "alchemi": ["magic"],
    "necromanc": ["magic"],
    "magic school": ["magic"],
    "cozy fantasi": ["fantasi", "lightheart"],
    "humorous fantasi": ["fantasi", "funni"],
    "animal fantasi": ["fantasi", "anim"],
    "christian fantasi": ["fantasi", "religion"],
    "religious fantasi": ["fantasi", "religion"],
    "fantasy litrpg": ["litrpg"],

    # --- Lore, Fable & Myth ---
    "fairy tale inspir": ["fairy tal"],
    "folk tal": ["fairy tal"],
    "cinderella": ["fairy tal"],
    "the sleeping beauti": ["fairy tal"],
    "beauty and the beast": ["fairy tal"],
    "mytholog": ["fairy tal"],
    "greek mytholog": ["fairy tal"],
    "norse mytholog": ["fairy tal"],
    "celtic": ["fairy tal"],
    "legend": ["fairy tal"],
    "folklor": ["fairy tal"],

    # --- Science Fiction & Space ---
    "sf": ["scifi"],
    "printsf": ["scifi"],
    "hard sf": ["scifi"],
    "science fantasi": ["scifi", "fantasi"],
    "interstellar travel": ["space travel"],
    "interplanetary voyag": ["space travel"],
    "space coloni": ["space travel"],
    "generation ship": ["space travel"],
    "space warfar": ["space travel", "militari"],
    "space pir": ["space travel", "adventur"],
    "moon": ["space travel"],
    "earth": ["space travel"],
    "mar": ["space travel"],
    "venu": ["space travel"],
    "asteroid": ["space travel"],
    "star trek the original seri": ["star trek"],
    "star war": ["space travel"],
    "doctor who fictitious charact": ["scifi"],
    "alien invas": ["alien"],
    "first contact": ["alien"],
    "alien artifact": ["alien"],
    "humanalien encount": ["alien"],
    "war with alien": ["alien"],
    "alien perspect": ["alien"],
    "martian": ["alien"],
    "extraterrestrial b": ["alien"],
    "dyson spher": ["scifi"],
    "terraform": ["scifi"],
    "antigrav": ["scifi"],
    "megaengin": ["scifi"],
    "technolog": ["scifi"],
    "scienc": ["scifi"],

    # --- Timeline & Reality Shifts ---
    "end of the world": ["apocalyps"],
    "cosy catastroph": ["apocalyps"],
    "disast": ["apocalyps"],
    "pandem": ["apocalyps"],
    "epidem": ["apocalyps"],
    "plagu": ["apocalyps"],
    "diseas": ["apocalyps"],
    "parallel world": ["parallel univers"],
    "transdimension": ["parallel univers"],
    "portalg": ["parallel univers"],
    "parallel univers": ["parallel univers"],
    "time loop": ["time travel"],
    "time slip": ["time travel"],
    "juvenile timetravel": ["time travel", "ya"],
    "hitler win": ["alternate histori"],
    "lost world": ["lost rac"],
    "lost rac": ["lost rac"],
    "atlanti": ["lost rac"],

    # --- Cyber, Tech & Gaming ---
    "computer gam": ["video gam"],
    "gamebook": ["video gam"],
    "game": ["video gam"],
    "virtual r": ["video gam"],
    "transhuman": ["cyberpunk"],
    "posthuman": ["cyberpunk"],
    "android": ["robot"],
    "cyborg": ["robot"],
    "artificial intellig": ["ai"],
    "comput": ["ai"],

    # --- Horror, Dark & Gothic ---
    "juvenile horror": ["ya", "horror"],
    "gothic horror": ["gothic"],
    "scari": ["horror"],
    "terror": ["horror"],
    "lovecraftian": ["horror"],
    "cannib": ["horror"],
    "bizarro": ["weird"],
    "haunted hous": ["ghost"],
    "ghost stori": ["ghost"],
    "youngadult ghost stori": ["ya", "ghost"],
    "afterlif": ["ghost"],

    # --- Paranormal Beings ---
    "shapeshift": ["shapechang"],
    "shifter": ["shapechang"],
    "lycanthropi": ["werewolf"],
    "dragon shift": ["shapechang", "dragon"],
    "youngadult supernatur": ["ya", "supernatur"],
    "occult detect": ["supernatur", "mysteri"],
    "paranorm": ["supernatur"],
    "paranormal": ["supernatur"],
    "occult": ["supernatur"],
    "voodoo": ["supernatur"],
    "demon": ["supernatur"],
    "angel": ["supernatur"],
    "satan": ["supernatur"],
    "devil": ["supernatur"],
    "hell": ["supernatur"],
    "possess": ["supernatur"],
    "vike": ["supernatur"], 
    "god": ["supernatur"],
    "immort": ["supernatur"],
    "longev": ["supernatur"],
    "reincarn": ["supernatur"],
    "resurrect": ["supernatur"],
    "vampir": ["vampir"],
    "werewolf": ["werewolf"],
    "zombi": ["zombi"],
    "dragon": ["dragon"],
    "ghost": ["ghost"],

    # --- Lesser Mythical Creatures ---
    "fairi": ["fairi"],
    "fa": ["fantasi"],
    "elf": ["fantasi"],
    "dwarf": ["fantasi"],
    "troll": ["fantasi"],
    "goblin": ["fantasi"],
    "gargoyl": ["fantasi"],
    "siren": ["fantasi"],
    "mermaid": ["fantasi"],
    "unicorn": ["fantasi"],
    "monster": ["horror"],
    "sea monst": ["horror"],
    "mythical creatur": ["fantasi"],

    # --- Romance Categories ---
    "rom": ["paranormal rom"],
    "love stori": ["paranormal rom"],
    "love": ["paranormal rom"],
    "fated m": ["paranormal rom"],
    "manwoman relationship": ["paranormal rom"],
    "erotic stori": ["erot"],
    "steami": ["erot"],
    "sex": ["erot"],
    "harem": ["erot"],
    "reverse harem": ["erot"],
    "enemies to lov": ["rom"],
    "slow burn": ["rom"],
    "forced proxim": ["rom"],
    "interpersonal rel": ["rom"],
    "marriag": ["rom"],

    # --- Mystery, Thriller & Action ---
    "thriller  suspens": ["thriller"],
    "suspens": ["thriller"],
    "technothril": ["thriller"],
    "detective and mystery stori": ["mysteri"],
    "murder mysteri": ["mysteri"],
    "private investig": ["mysteri"],
    "detect": ["mysteri"],
    "crime": ["mysteri"],
    "spy": ["mysteri"],
    "spi": ["mysteri"],
    "espionag": ["mysteri"],
    "assassin": ["mysteri"],
    "conspiraci": ["thriller"],
    "intrigu": ["thriller"],
    "kidnap": ["thriller"],
    "serial kil": ["thriller"],
    "missing person": ["mysteri"],
    "adventure stori": ["adventur"],
    "adventure and adventur": ["adventur"],
    "fantastic adventur": ["adventur"],
    "planetary adventur": ["adventur"],
    "sea adventur": ["adventur"],
    "pirat": ["adventur"],
    "escap": ["adventur"],
    "rescu": ["adventur"],
    "surviv": ["adventur"],
    "explor": ["adventur"],

    # --- Psychic & Mental Powers ---
    "telepathi": ["psy pow"],
    "precognit": ["psy pow"],
    "psychic": ["psy pow"],
    "clairvoy": ["psy pow"],
    "esp": ["psy pow"],
    "telekinesi": ["psy pow"],
    "mind control": ["psy pow"],
    "body swap": ["psy pow"],
    "teleport": ["psy pow"],
    "invis": ["psy pow"],
    "memori": ["psy pow"],
    "amnesium": ["psy pow"],

    # --- Biology & Engineering ---
    "clone": ["genetic engin"],
    "mutant": ["genetic engin"],
    "evolut": ["genetic engin"],
    "cryogen": ["genetic engin"],
    "suspended anim": ["genetic engin"],
    "scientist": ["genetic engin"],

    # --- Demographics & Representation ---
    "female protagonist": ["female lead"],
    "female main charact": ["female lead"],
    "strong female charact": ["female lead"],
    "female warrior": ["female lead"],
    "woman": ["female lead"],
    "written by woman": ["female lead"],
    "gay": ["lgbtq"],
    "lesbian": ["lgbtq"],
    "queer": ["lgbtq"],
    "bisexu": ["lgbtq"],
    "transgend": ["lgbtq"],
    "nonbinary gend": ["lgbtq"],
    "sapphic": ["lgbtq"],
    "gay rom": ["lgbtq", "paranormal rom"],

    # --- Settings, Regions & History ---
    "united st": ["american"],
    "new york c": ["american"],
    "californium": ["american"],
    "san francisco": ["american"],
    "new orlean": ["american"],
    "native american": ["american"],
    "african american": ["american"],
    "london": ["england"],
    "great britain": ["england"],
    "uk": ["england"],
    "scotland": ["england"],
    "ireland": ["england"],
    "franc": ["contemporari"],
    "china": ["contemporari"],
    "japan": ["contemporari"],
    "egypt": ["contemporari"],
    "africa": ["contemporari"],
    "south africa": ["contemporari"],
    "mexico": ["contemporari"],
    "canada": ["contemporari"],
    "canadian fict": ["contemporari"],
    "western": ["histori"],
    "weird west": ["histori"],
    "victorian": ["histori"],
    "noir": ["histori"],
    "cold war": ["histori"],
    "civil war": ["histori"],
    "world war iu": ["histori"],
    "militari": ["war"],
    "military fantasi": ["war"],
    "imaginary wars and battl": ["war"],
    "interstellar war": ["war"],
    "revolut": ["war"],
    "rebellion": ["war"],
    "polit": ["war"],
    "empir": ["war"],

    # --- Narrative Tone & Literary Tropes ---
    "satir": ["funni"],
    "parodi": ["funni"],
    "comic": ["funni"],
    "hummi": ["funni"],
    "insan": ["sad"],
    "suicid": ["sad"],
    "death": ["sad"],
    "reveng": ["sad"],
    "betray": ["sad"],
    "good and evil": ["challeng"],
    "social commentari": ["challeng"],
    "social critic": ["challeng"],
    "social cla": ["challeng"],
    "racism": ["challeng"],
    "slaveri": ["challeng"],
    "genocid": ["challeng"],
    "surreal": ["reflect"],
    "philosophi": ["reflect"],
    "psycholog": ["reflect"],
    "literari": ["reflect"],
    "coming of ag": ["friendship"],
    "famili": ["friendship"],
    "sibl": ["friendship"],
    "brother": ["friendship"],
    "brothers and sist": ["friendship"],
    "twin": ["friendship"],
    "orphan": ["friendship"],
    "child": ["childrens stori"],
    "teenag": ["ya"],
    "boarding school": ["ya"],
    "magic school": ["fantasi"],

    # --- General / Common Word Long Tail ---
    "cat": ["anim"],
    "dog": ["anim"],
    "hors": ["anim"],
    "rat": ["anim"],
    "mous": ["anim"],
    "spider": ["anim"],
    "dolphin": ["anim"],
    "dinosaur": ["anim"],
    "insect": ["anim"],
    "talking anim": ["anim", "fairy tal"],
    "anthropomorph": ["anim"],
    "forest": ["imaginary plac"],
    "desert": ["imaginary plac"],
    "water": ["imaginary plac"],
    "sea": ["imaginary plac"],
    "small town": ["imaginary plac"],
    "circu": ["imaginary plac"],
    "librari": ["imaginary plac"],
    "music": ["art"],
    "poetri": ["art"],
    "food": ["art"],
    "travel": ["adventur"],
    "explor": ["adventur"],
}

In [836]:
tag_mapping_final = {
    # --- Sci-Fi, Space & Technology ---
    "science fict": ["sf"],
    "scienc fict": ["sf"],
    "sciense fict": ["sf"],
    "scifi": ["sf"],
    "printsf": ["sf"],
    "hard sf": ["sf", "hard sf"],
    "hard science fict": ["sf", "hard sf"],
    "science fiction  fantasi": ["sf", "fantasi"],
    "science fiction fantasi": ["sf", "fantasi"],
    "science fantasi": ["sf", "fantasi"],
    "sf romanc": ["sf", "rom"],
    "scifi rom": ["sf", "rom"],
    "sf rom": ["sf", "rom"],
    "military sf": ["militari", "sf"],
    "juvenile sf": ["ya", "sf"],
    "youngadult sf": ["ya", "sf"],
    "teen science fict": ["ya", "sf"],
    "middle grades science fict": ["ya", "sf"],
    "speculative fict": ["sf"],
    "space st": ["space"],
    "space opu": ["space"],
    "interplanetary travel": ["space travel"],
    "interplanetary voyag": ["space travel"],
    "space coloni": ["space travel"],
    "generation ship": ["space travel"],
    "space warfar": ["space travel", "militari"],
    "space pir": ["space travel", "adventur"],
    "star war": ["space travel"],
    "star trek the original seri": ["star trek"],
    "star trek": ["space travel"],
    "doctor who fictitious charact": ["sf"],
    "alien invas": ["alien"],
    "first contact": ["alien"],
    "alien artifact": ["alien"],
    "humanalien encount": ["alien"],
    "war with alien": ["alien"],
    "alien perspect": ["alien"],
    "martian": ["alien"],
    "extraterrestrial b": ["alien"],
    "aliens": ["alien"],
    "dyson spher": ["sf"],
    "terraform": ["sf"],
    "antigrav": ["sf"],
    "megaengin": ["sf"],
    "technolog": ["sf"],
    "nanotechnolog": ["sf"],
    "scienc": ["sf"],
    "climate chang": ["apocalyps"],
    "ecolog": ["apocalyps"],
    "post apocalypt": ["apocalyps"],
    "postapocalypt": ["apocalyps"],
    "apocalypt": ["apocalyps"],
    "end of the world": ["apocalyps"],
    "cosy catastroph": ["apocalyps"],
    "disast": ["apocalyps"],
    "pandem": ["apocalyps"],
    "epidem": ["apocalyps"],
    "plagu": ["apocalyps"],
    "diseas": ["apocalyps"],

    # --- Fantasy Elements & Mythos ---
    "fantasy": ["fantasi"],
    "fantezi": ["fantasi"],
    "fantasy fict": ["fantasi"],
    "epic": ["epic fantasi"],
    "high fantasi": ["epic fantasi"],
    "heroic fantasi": ["epic fantasi"],
    "quest": ["epic fantasi"],
    "progression fantasi": ["epic fantasi"],
    "magician": ["magic"],
    "wizard": ["magic"],
    "witch": ["magic"],
    "druid": ["magic"],
    "alchemi": ["magic"],
    "necromanc": ["magic"],
    "magic school": ["magic"],
    "cozy fantasi": ["fantasi", "lightheart"],
    "humorous fantasi": ["fantasi", "funni"],
    "fantasyhummi": ["fantasi"],
    "christian fantasi": ["fantasi", "religion"],
    "religious fantasi": ["fantasi", "religion"],
    "fantasy litrpg": ["litrpg"],
    "faeri": ["fairi"],
    "fairi": ["fantasi"],
    "fa": ["fantasi"],
    "elf": ["fantasi"],
    "dwarf": ["fantasi"],
    "troll": ["fantasi"],
    "goblin": ["fantasi"],
    "gargoyl": ["fantasi"],
    "siren": ["fantasi"],
    "mermaid": ["fantasi"],
    "unicorn": ["fantasi"],
    "mythical creatur": ["fantasi"],
    "dragons  mythical creatur": ["dragon", "fairy tal"],
    "dragonl": ["dragon"],
    "fairy tale inspir": ["fairy tal"],
    "folk tal": ["fairy tal"],
    "cinderella": ["fairy tal"],
    "the sleeping beauti": ["fairy tal"],
    "beauty and the beast": ["fairy tal"],
    "mytholog": ["fairy tal"],
    "greek mytholog": ["fairy tal"],
    "norse mytholog": ["fairy tal"],
    "celtic": ["fairy tal"],
    "legend": ["fairy tal"],
    "folklor": ["fairy tal"],
    "retold fairy tal": ["fantasi"],
    "retel": ["fairy tal"],
    "mythical": ["fairy tal"],
    "discworld": ["fantasi"],
    "propheci": ["fantasi"],
    "secret": ["fantasi"],

    # --- Alternative Realities & Timelines ---
    "parallel world": ["parallel univers"],
    "transdimension": ["parallel univers"],
    "portalg": ["parallel univers"],
    "time loop": ["time travel"],
    "time slip": ["time travel"],
    "time travel rom": ["rom", "time travel"],
    "timetravel rom": ["time travel", "rom"],
    "hitler win": ["alternate histori"],
    "alternative histories fict": ["alternate histori"],
    "alternate world": ["alternate histori"],
    "secret histori": ["alternate histori"],
    "alternate univers": ["alternate histori"],
    "lost world": ["lost rac"],
    "lost rac": ["lost rac"],
    "atlanti": ["lost rac"],

    # --- Cyber, Tech & Gaming ---
    "computer gam": ["video gam"],
    "gamebook": ["video gam"],
    "game": ["video gam"],
    "virtual r": ["video gam"],
    "transhuman": ["cyberpunk"],
    "posthuman": ["cyberpunk"],
    "android": ["robot"],
    "cyborg": ["robot"],
    "artificial intellig": ["ai"],
    "comput": ["ai"],

    # --- Horror, Ghosts & Dark Themes ---
    "horreur": ["horror"],
    "read horror": ["horror"],
    "body horror": ["horror"],
    "horror tal": ["horror"],
    "gothic horror": ["gothic"],
    "scari": ["horror"],
    "terror": ["horror"],
    "lovecraftian": ["horror"],
    "cannib": ["horror"],
    "monster": ["horror"],
    "sea monst": ["horror"],
    "bizarro": ["weird"],
    "haunted hous": ["ghost"],
    "ghost stori": ["ghost"],
    "afterlif": ["ghost"],

    # --- Paranormal & Shifters ---
    "paranormal  urban": ["supernatur"],
    "paranormal powers": ["supernatur"],
    "paranormal mystery": ["supernatur"],
    "paranormal mysteri": ["supernatur"],
    "paranormal thril": ["supernatur"],
    "paranormal fict": ["supernatur"],
    "paranormal pow": ["supernatur"],
    "paranorm": ["supernatur"],
    "paranormal": ["supernatur"],
    "supernatural thril": ["supernatur"],
    "occult detect": ["supernatur", "mysteri"],
    "shapeshift": ["shapechang"],
    "shifter": ["shapechang"],
    "lycanthropi": ["werewolf"],
    "dragon shift": ["shapechang", "dragon"],
    "voodoo": ["supernatur"],
    "demonolog": ["demon"],
    "satan": ["supernatur"],
    "devil": ["supernatur"],
    "hell": ["supernatur"],
    "possess": ["supernatur"],
    "vike": ["supernatur"],
    "god": ["supernatur"],
    "longev": ["supernatur"],
    "reincarn": ["supernatur"],
    "resurrect": ["supernatur"],

    # --- Romance Categorization ---
    "romanc": ["rom"],
    "romant": ["rom"],
    "love stori": ["paranormal rom"],
    "love": ["paranormal rom"],
    "fated m": ["paranormal rom"],
    "manwoman relationship": ["paranormal rom"],
    "paranormal romance stori": ["paranormal rom"],
    "paranormal romance stories": ["paranormal rom"],
    "paranormal erotic romance": ["paranormal rom"],
    "paranormal erotic rom": ["paranormal rom"],
    "fantasyrom": ["fantasy rom"],
    "romantic fantasi": ["fantasy rom"],
    "romantasi": ["fantasy rom"],
    "vampire rom": ["vampir", "rom"],
    "dark rom": ["dark", "rom"],
    "mm romanc": ["lgbtq"],
    "erotica": ["erot"],
    "gay erotica": ["lgbtq", "erot"],
    "erotic stori": ["erot"],
    "steami": ["erot"],
    "sex": ["erot"],
    "harem": ["erot"],
    "reverse harem": ["erot"],
    "enemies to lov": ["rom"],
    "slow burn": ["rom"],
    "forced proxim": ["rom"],
    "interpersonal rel": ["rom"],
    "marriag": ["rom"],

    # --- Mystery, Thriller & Action ---
    "thriller  suspens": ["thriller"],
    "suspens": ["thriller"],
    "suspenseful": ["thriller"],
    "suspense": ["thriller"],
    "technothril": ["thriller"],
    "conspiraci": ["thriller"],
    "intrigu": ["thriller"],
    "kidnap": ["thriller"],
    "serial kil": ["thriller"],
    "abduct": ["thriller"],
    "paranormal suspens": ["supernatur", "thriller"],
    "thriller  suspensescience fiction  fantasi": ["thriller", "sf", "fantasi"],
    "detective and mystery stori": ["mysteri"],
    "murder mysteri": ["mysteri"],
    "murder": ["mysteri"],
    "private investig": ["mysteri"],
    "detect": ["mysteri"],
    "crime": ["mysteri"],
    "spy": ["mysteri"],
    "spi": ["mysteri"],
    "espionag": ["mysteri"],
    "assassin": ["mysteri"],
    "missing person": ["mysteri"],
    "cozy mysteri": ["mysteri", "lightheart"],
    "futuristic mysteri": ["near futur", "mysteri"],
    "sherlock holm": ["mysteri"],
    "action  adventur": ["adventur"],
    "actionadventur": ["adventur"],
    "action": ["adventur"],
    "aventur": ["adventur"],
    "aventure": ["adventur"],
    "adventurers": ["adventur"],
    "aventura": ["adventur"],
    "adventurous": ["adventur"],
    "adventure plot": ["adventur"],
    "misadventures": ["adventur"],
    "adventures": ["adventur"],
    "adventure stori": ["adventur"],
    "adventure and adventur": ["adventur"],
    "fantastic adventur": ["adventur"],
    "planetary adventur": ["adventur"],
    "sea adventur": ["adventur"],
    "pirat": ["adventur"],
    "escap": ["adventur"],
    "rescu": ["adventur"],
    "surviv": ["adventur"],
    "explor": ["adventur"],
    "travel": ["adventur"],
    "martial art": ["adventur"],
    "choose your own adventur": ["adventur"],
    "fix up": ["adventur"],
    "standalon": ["adventur"],
    "book": ["adventur"],
    "short stori": ["adventur"],

    # --- Pacing and Perspective Formats ---
    "slow burn": ["slowpac"],
    "first person point of view": ["first person"],

    # --- Dystopia Pole Realignment ---
    "dystopia": ["dystopian"],
    "dystopias": ["dystopian"],
    "dystopium": ["dystopian"],
    "utopium": ["dystopian"],

    # --- Psychic & Biological Elements ---
    "psychic ": ["psy pow"],
    "psychic": ["psy pow"],
    "telepathi": ["psy pow"],
    "precognit": ["psy pow"],
    "clairvoy": ["psy pow"],
    "esp": ["psy pow"],
    "telekinesi": ["psy pow"],
    "mind control": ["psy pow"],
    "body swap": ["psy pow"],
    "teleport": ["psy pow"],
    "invis": ["psy pow"],
    "memori": ["psy pow"],
    "amnesium": ["psy pow"],
    "clone": ["genetic engin"],
    "mutant": ["genetic engin"],
    "evolut": ["genetic engin"],
    "cryogen": ["genetic engin"],
    "suspended anim": ["genetic engin"],
    "scientist": ["genetic engin"],
    "mad scientist": ["genetic engin"],
    "drug": ["genetic engin"],

    # --- Character Descriptors & Identity ---
    "strong female characters": ["female lead"],
    "strong female charact": ["female lead"],
    "female protagonist": ["female lead"],
    "female main charact": ["female lead"],
    "female warrior": ["female lead"],
    "woman": ["female lead"],
    "written by woman": ["female lead"],
    "lgbt": ["lgbtq"],
    "gay": ["lgbtq"],
    "lesbian": ["lgbtq"],
    "queer": ["lgbtq"],
    "bisexu": ["lgbtq"],
    "transgend": ["lgbtq"],
    "nonbinary gend": ["lgbtq"],
    "sapphic": ["lgbtq"],

    # --- Demographics & Age Groups ---
    "young adult": ["ya"],
    "youngadult": ["ya"],
    "teen  young adult": ["ya"],
    "youngadult fict": ["ya"],
    "juvenile fict": ["ya"],
    "young adult fict": ["ya"],
    "juvenil": ["ya"],
    "teenag": ["ya"],
    "teen": ["ya"],
    "boarding school": ["ya"],
    "young adult fantasi": ["ya", "fantasi"],
    "youngadultfantasi": ["ya", "fantasi"],
    "youngadult fantasi": ["ya", "fantasi"],
    "teen fantasi": ["ya", "fantasi"],
    "juvenile fantasi": ["ya", "fantasi"],
    "teen urban fantasi": ["ya", "urban fantasi"],
    "juvenile urban fantasi": ["ya", "urban fantasi"],
    "youngadult animal fantasi": ["ya", "animal fantasi"],
    "young adult urban fantasi": ["ya", "urban fantasi"],
    "youngadult historical fantasi": ["ya", "historical fantasi"],
    "teen historical fantasi": ["ya", "historical fantasi"],
    "youngadult horror": ["ya", "horror"],
    "youngadult vampir": ["ya", "vampir"],
    "youngadult timetravel": ["ya", "time travel"],
    "juvenile timetravel": ["time travel", "ya"],
    "child": ["childrens stori"],
    "na driven": ["adult"],

    # --- Settings, Regions & Cultural History ---
    "american fict": ["american"],
    "united st": ["american"],
    "new york c": ["american"],
    "californium": ["american"],
    "san francisco": ["american"],
    "new orlean": ["american"],
    "native american": ["american"],
    "african american": ["american"],
    "london": ["england"],
    "great britain": ["england"],
    "uk": ["england"],
    "scotland": ["england"],
    "ireland": ["england"],
    "english fict": ["england"],
    "english": ["england"],
    "franc": ["contemporari"],
    "china": ["contemporari"],
    "japan": ["contemporari"],
    "egypt": ["contemporari"],
    "africa": ["contemporari"],
    "south africa": ["contemporari"],
    "mexico": ["contemporari"],
    "canada": ["contemporari"],
    "canadian fict": ["contemporari"],
    "thailand": ["contemporari"],
    "antarctica": ["contemporari"],
    "australium": ["contemporari"],
    "future australium": ["contemporari"],
    "indium": ["contemporari"],
    "spanish": ["contemporari"],
    "african": ["contemporari"],
    "african speculative fict": ["contemporari"],
    "histor": ["histori"],
    "historical fict": ["histori"],
    "prehistor": ["histori"],
    "western": ["histori"],
    "weird west": ["histori"],
    "victorian": ["histori"],
    "noir": ["histori"],
    "cold war": ["histori"],
    "civil war": ["histori"],
    "world war iu": ["histori"],
    "historical fantasy rom": ["historical fantasi", "fantasy rom"],
    "arthurian fantasi": ["historical fantasi"],
    "arthurian rom": ["historical fantasi", "rom"],
    "archeolog": ["histori"],
    "archaeolog": ["histori"],
    "militari": ["war"],
    "military fantasi": ["war"],
    "imaginary wars and battl": ["war"],
    "interstellar war": ["war"],
    "revolut": ["war"],
    "rebellion": ["war"],
    "polit": ["war"],
    "empir": ["war"],
    "colon": ["space travel"],
    "lost coloni": ["space travel"],

    # --- Narrative Tone & Themes ---
    "humor": ["funni"],
    "satir": ["funni"],
    "parodi": ["funni"],
    "comic": ["funni"],
    "hummi": ["funni"],
    "insan": ["sad"],
    "suicid": ["sad"],
    "death": ["sad"],
    "reveng": ["sad"],
    "betray": ["sad"],
    "cold": ["sad"],
    "alcohol": ["sad"],
    "good and evil": ["challeng"],
    "social commentari": ["challeng"],
    "social critic": ["challeng"],
    "social cla": ["challeng"],
    "racism": ["challeng"],
    "slaveri": ["challeng"],
    "genocid": ["challeng"],
    "cur": ["challeng"],
    "surreal": ["reflect"],
    "philosophi": ["reflect"],
    "psycholog": ["reflect"],
    "literature  fiction": ["reflect"],
    "literature  fict": ["reflect"],
    "literatur": ["reflect"],
    "literary fict": ["reflect"],
    "literary fantasi": ["reflect", "fantasi"],
    "metafict": ["reflect"],
    "dream": ["reflect"],
    "fate": ["reflect"],
    "coming of ag": ["friendship"],
    "famili": ["friendship"],
    "sibl": ["friendship"],
    "brother": ["friendship"],
    "brothers and sist": ["friendship"],
    "twin": ["friendship"],
    "orphan": ["friendship"],
    "christian fict": ["religion"],
    "christian": ["religion"],
    "futuristic": ["near futur"],
    "far futur": ["near futur"],
    "distant futur": ["near futur"],
    "futuristic rom": ["near futur", "rom"],
    "wish": ["hope"],
    "cozy mysteri": ["mysteri", "lightheart"],

    # --- Animals & Core Habitats ---
    "cat": ["anim"],
    "dog": ["anim"],
    "hors": ["anim"],
    "rat": ["anim"],
    "mous": ["anim"],
    "spider": ["anim"],
    "dolphin": ["anim"],
    "dinosaur": ["anim"],
    "insect": ["anim"],
    "talking anim": ["anim", "fairy tal"],
    "anthropomorph": ["anim"],
    "selki": ["anim"],
    "forest": ["imaginary plac"],
    "desert": ["imaginary plac"],
    "water": ["imaginary plac"],
    "sea": ["imaginary plac"],
    "small town": ["imaginary plac"],
    "circu": ["imaginary plac"],
    "librari": ["imaginary plac"],
    "hollow earth": ["imaginary plac"],
    "basebal": ["imaginary plac"],
    "sport": ["imaginary plac"],

    # --- Arts, Media & Fallbacks ---
    "music": ["art"],
    "poetri": ["art"],
    "food": ["art"],
    "invent": ["genetic engin"],
}

In [819]:
for threshold in [5, 10, 20, 50]:
    print(threshold, (tag_counts >= threshold).sum())

5 1963
10 1187
20 727
50 389


In [837]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

def tag_cleanup(taglist):
    clean_tags = []
    if isinstance(taglist, float):
        return taglist
    # taglist = [stemmer.stem(word) for word in taglist]
    # taglist = [lemmatizer.lemmatize(word) for word in taglist]
    for tag in taglist:
        if tag not in valid_tags:
            continue
        canonical_tag = tag_mapping.get(tag, tag)
        if canonical_tag is not None:
            if isinstance(canonical_tag, list):
                clean_tags.extend(canonical_tag)
            else:
                clean_tags.append(canonical_tag)
    return list(set(clean_tags)) 


In [838]:
books['tags_clean'] = books["tags"].apply(tag_cleanup)

tag_clean_counts = books['tags'].explode().value_counts()


In [839]:
books['tags_clean']

0                             [american, horror, apocalyps]
1                                                       NaN
2                                                       NaN
3                                                       NaN
4                                                [sf, anim]
                                ...                        
151586                              [fantasi, ya, adventur]
151587    [lgbtq, emot, sad, diverse charact, character ...
151588                                                   []
151589                                             [sf, ya]
151590    [loveable charact, lgbtq, not diverse charact,...
Name: tags_clean, Length: 151591, dtype: object

In [840]:
for threshold in [5, 10, 20, 50]:
    print(threshold, (tag_clean_counts >= threshold).sum())

5 1963
10 1187
20 727
50 389


In [842]:
books['tags_clean'].explode().value_counts()#.index.tolist()

tags_clean
fantasi                    47230
ya                         28502
sf                         26718
adventur                   14759
paranormal rom              7533
                           ...  
drama                         20
linguist                      20
tiein                         20
multiple points of view       20
teen                          20
Name: count, Length: 151, dtype: int64

In [843]:
books['tags_clean'].explode().value_counts().to_csv("list_tags_final.csv")

In [733]:
tagsv.index.tolist()

['ya',
 '',
 'horror',
 'dystopian',
 'a mix driven',
 'dark',
 'space',
 'youngadult sf',
 'lgbtq',
 'war',
 'plot driven',
 'time travel',
 'character driven',
 'juvenile sf',
 'thriller',
 'sad',
 'space opera',
 'dystopias',
 'magic',
 'ghost',
 'werewolf',
 'sf',
 'military',
 'youngadult horror',
 'steampunk',
 'ofearnaseriesnovel',
 'humour',
 'superhero',
 'project gutenberg',
 'murder',
 'black author',
 'litrpg',
 'female protagonist',
 'novel',
 'juvenile horror',
 'erotica',
 'gay',
 'action',
 'religion',
 'occult',
 'read',
 'lesbian',
 'friendship',
 'first contact',
 'librivox',
 'ofearnaread',
 'crime',
 'gothic',
 'cyberpunk',
 'epic',
 'american',
 'interstellar travel',
 'quest',
 'ofearnanovel',
 'assassin',
 'england',
 'hard sf',
 'scifi',
 'nesfa core reading list',
 'gollancz sf masterwork',
 'first person point of view',
 'london',
 'big sky library  to find',
 'fae',
 'weird west',
 'adult',
 'psychic',
 'gay erotica',
 'star trek',
 'english',
 'noir',
 'mus

In [734]:
# Assuming df_exploded is your exploded DataFrame
tagsv = books['tags_clean'].explode().value_counts()

# Calculate cumulative percentage
tag_pct = tagsv.cumsum() / tagsv.sum()

# See how many tags make up 80% of your total data
essential_tags_count = (tag_pct <= 0.80).sum()
print(f"Just {essential_tags_count} tags make up 80% of the entire dataset.")

Just 45 tags make up 80% of the entire dataset.


### Tags

In [43]:
import string
from sklearn.feature_extraction.text import TfidfVectorizer


test_tags = books_eda['tags'][~books_eda['tags'].isna()]

test_tags = [item.translate(str.maketrans('', '', string.punctuation)).lower() for item in test_tags]
test_tags

['anatomy of wonder 1 core collection biological warfare cannibalism famine nuclear warfare postapocalyptic postapocalyptic united states viral diseases',
 'insects librivox project gutenberg science fiction spiders',
 'science fiction',
 'aliens diplomacy earth first contact juvenile sf pets science fiction young adult youngadult sf',
 'nesfa core reading list science fiction',
 'classics adventure fiction science fiction comics literature',
 '1001 books you must read before you die anatomy of wonder 1 core collection censorship dystopia fantasy librivox project gutenberg utopia',
 'aliens california chickens first contact fungi interplanetary travel juvenile sf mushrooms',
 'mystery space opera',
 'abebooks 50 essential sf books alternate worlds antigravity architecture artificial bodies cars durable goods economics genetic memory hypnosis illinois implanted memories interstellar communication longevity manhunt mutants new jersey new york city nostalgia paranormal abilities pioneers 

In [61]:
books_eda['tags'][0]

"['Anatomy of Wonder 1 Core Collection', 'biological warfare', 'cannibalism', 'famine', 'nuclear warfare', 'post-apocalyptic', 'postapocalyptic', 'United States', 'viral diseases']"

In [44]:
tfidf = TfidfVectorizer()
result = tfidf.fit_transform(test_tags)

In [45]:
print('\nidf values:')
for ele1, ele2 in zip(tfidf.get_feature_names_out(), tfidf.idf_):
    print(ele1, ':', ele2)


idf values:
05 : 8.712220195674035
100 : 3.9415355712083704
1001 : 4.928030561755774
110 : 8.712220195674035
114 : 8.712220195674035
117 : 8.712220195674035
119 : 8.712220195674035
122 : 8.712220195674035
128 : 8.712220195674035
129 : 8.712220195674035
1300s : 8.712220195674035
1735854180417 : 8.712220195674035
1735854344357 : 8.712220195674035
1735865543602 : 8.712220195674035
1735865548048 : 8.712220195674035
1747106705384 : 8.712220195674035
1753212335831 : 8.712220195674035
17th : 8.712220195674035
1800s : 8.712220195674035
1900 : 7.459457227178667
1900s : 8.712220195674035
1930s : 8.306755087565872
19391945 : 8.306755087565872
19391984 : 6.40963510267999
1940s : 8.712220195674035
1944 : 7.79592946379988
1945 : 7.6136079070059255
19461987 : 5.171260871636721
19491984 : 4.905557705903716
1950 : 6.69731717513177
1950s : 8.01907301511409
1954 : 8.712220195674035
1959 : 8.712220195674035
1960s : 6.227313545886035
19611975 : 8.712220195674035
1967 : 8.712220195674035
1968 : 8.712220195

In [62]:
from nltk.stem import WordNetLemmatizer

first, cleanup for the tags

In [46]:
all_finalist_tags = [[item.strip().strip("'").lower() for item in taglist.strip('[]').split(',')]
    for taglist in finalists['tags'].tolist()
    if isinstance(taglist, str)]

flat_list_f = pd.Series([tag for taglist in all_finalist_tags for tag in taglist])

all_nf_tags = [[item.strip().strip("'").lower() for item in taglist.strip('[]').split(',')]
    for taglist in nonfinalists['tags'].tolist()
    if isinstance(taglist, str)]

flat_list_nf = pd.Series([tag for taglist in all_nf_tags for tag in taglist])

In [63]:
print(flat_list_f)

0                            1-award-winner
1           abebooks: 50 essential sf books
2       anatomy of wonder 1 core collection
3                                 detective
4                                   earworm
                       ...                 
4629                        science fiction
4630                               classics
4631                                fantasy
4632                              adventure
4633                                  space
Length: 4634, dtype: str


In [47]:
flat_list_f.value_counts()

science fiction                       410
fantasy                               230
fiction                               144
hugo award for best novel finalist    106
classics                              105
                                     ... 
gothic                                  1
werewolf                                1
brothers                                1
crossover                               1
set at specfic conventions              1
Name: count, Length: 1153, dtype: int64

In [48]:
flat_list_nf.value_counts()

science fiction      1913
fantasy              1232
fiction               922
classics              562
adventure             530
                     ... 
selling your soul       1
the blitz               1
boredom                 1
evacuation              1
homeless persons        1
Name: count, Length: 2773, dtype: int64

In [163]:
flat_list2_f = flat_list_f[~flat_list_f.str.contains('award')]
# remove all the ones which say they won an award...

flat_list2_nf = flat_list_nf[~flat_list_nf.str.contains('award')]

flat_list2_nf = flat_list_nf[~flat_list_nf.str.contains('best')]

flat_list2_nf = flat_list_nf[~flat_list_nf.str.contains('list')]


flat_list2_nf = flat_list_nf[~flat_list_nf.str.contains('movie')]


flat_list2_nf = flat_list_nf[~flat_list_nf.str.contains('collection')]

import re

flat_list2_nf = flat_list_nf[~flat_list_nf.str.contains(r"\d", na=False)]

print(flat_list2_nf.value_counts().to_string())

science fiction                                            1913
fantasy                                                    1232
fiction                                                     922
classics                                                    562
adventure                                                   530
                                                            486
project gutenberg                                           323
horror                                                      280
aliens                                                      267
young adult                                                 254
juvenile fantasy                                            186
dystopian                                                   185
librivox                                                    167
ofearna-ebooks                                              164
time travel                                                 150
young-adult fantasy                     

In [164]:
nltk_test = flat_list2_nf[0:100].to_list()#books_eda['tags'][0].translate(str.maketrans('', '', string.punctuation)).lower()

# test_tags = [item.translate(str.maketrans('', '', string.punctuation)).lower() for item in test_tags]


In [109]:
from nltk.tokenize import word_tokenize
tokens = [word_tokenize(word) for word in nltk_test]
tokens = [item for sublist in tokens for item in sublist]

In [110]:
tokens

['biological',
 'warfare',
 'cannibalism',
 'famine',
 'nuclear',
 'warfare',
 'post-apocalyptic',
 'postapocalyptic',
 'united',
 'states',
 'viral',
 'diseases',
 'insects',
 'librivox',
 'project',
 'gutenberg',
 'science',
 'fiction',
 'spiders',
 'science',
 'fiction',
 'aliens',
 'diplomacy',
 'earth',
 'first',
 'contact',
 'juvenile',
 'sf',
 'pets',
 'science',
 'fiction',
 'young',
 'adult',
 'young-adult',
 'sf',
 'nesfa',
 'core',
 'reading',
 'list',
 'science',
 'fiction',
 'classics',
 'adventure',
 'fiction',
 'science',
 'fiction',
 'comics',
 'literature',
 'censorship',
 'dystopia',
 'fantasy',
 'librivox',
 'project',
 'gutenberg',
 'utopia',
 'aliens',
 'california',
 'chickens',
 'first',
 'contact',
 'fungi',
 'interplanetary',
 'travel',
 'juvenile',
 'sf',
 'mushrooms',
 'mystery',
 'space',
 'opera',
 'alternate',
 'worlds',
 'antigravity',
 'architecture',
 'artificial',
 'bodies',
 'cars',
 'durable',
 'goods',
 'economics',
 'genetic',
 'memory',
 'hypnosis

In [111]:
len(np.unique(np.array(tokens)))

108

In [114]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
tokens = [stemmer.stem(word) for word in tokens]
print(tokens)

['biolog', 'warfar', 'cannib', 'famin', 'nuclear', 'warfar', 'post-apocalypt', 'postapocalypt', 'unit', 'state', 'viral', 'disea', 'insect', 'librivox', 'project', 'gutenberg', 'scienc', 'fiction', 'spider', 'scienc', 'fiction', 'alien', 'diplomaci', 'earth', 'first', 'contact', 'juvenil', 'sf', 'pet', 'scienc', 'fiction', 'young', 'adult', 'young-adult', 'sf', 'nesfa', 'core', 'read', 'list', 'scienc', 'fiction', 'classic', 'adventur', 'fiction', 'scienc', 'fiction', 'comic', 'literatur', 'censorship', 'dystopia', 'fantasi', 'librivox', 'project', 'gutenberg', 'utopia', 'alien', 'california', 'chicken', 'first', 'contact', 'fungi', 'interplanetari', 'travel', 'juvenil', 'sf', 'mushroom', 'mysteri', 'space', 'opera', 'altern', 'world', 'antigrav', 'architectur', 'artifici', 'bodi', 'car', 'durabl', 'good', 'econom', 'genet', 'memori', 'hypnosi', 'illinoi', 'implant', 'memori', 'interstellar', 'commun', 'longev', 'manhunt', 'mutant', 'new', 'jersey', 'new', 'york', 'citi', 'nostalgia', 

In [115]:
len(np.unique(np.array(tokens)))

105

In [116]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
tokens = [lemmatizer.lemmatize(word) for word in tokens]

In [117]:
len(np.unique(np.array(tokens)))

105

In [ ]:
# import nltk 
# nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to /Users/yilda/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

how about tf-idf

In [97]:
# flat_list2 = flat_list2[~((flat_list2 == 'fantasy') | (flat_list2 == 'fiction') | (flat_list2 == 'science fiction'))]


In [98]:
flat_list2_f.value_counts()

science fiction     2226
fantasy             1776
fiction             1150
adventure            710
adventurous          522
                    ... 
liminal                1
thoughtful             1
healing                1
single pov             1
first person pov       1
Name: count, Length: 2819, dtype: int64

In [99]:
flat_list2_nf.value_counts()

fantasy                       36468
science fiction               23551
fiction                       19613
adventure                     11512
young adult                    9993
                              ...  
gross                             1
aardvark                          1
fiction / horror / general        1
irish folk                        1
rover                             1
Name: count, Length: 8399, dtype: int64